# XGBoost Binary Model (Top 10 Features)

This notebook restricts the feature space to the ten engineered columns shown in the earlier screenshot while keeping the same preprocessing and evaluation logic as the standalone script.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from xgboost import XGBClassifier

DATA_PATH = Path("T49.2_Sep2025_1_StGallen.csv")
TARGET_COLUMN = "OUTCOME_3Kat_KHK"

SELECTED_CATEGORICAL = [
    "ANY_OPO",
    "Osteoporos_inkl_Osteopenie_T_Wert_abhängig",
    "ANY_OPE",
    "currsmo0",
]

SELECTED_NUMERIC = [
    "Cer_d18_16",
    "Cer_d18_24_0",
    "GESAMT_A",
    "CERT2",
]

TOP_TRANSFORMED_FEATURES = [
    "cat__ANY_OPO_1.0",
    "cat__Osteoporos_inkl_Osteopenie_T_Wert_abhängig_1.0",
    "cat__ANY_OPO_0.0",
    "cat__ANY_OPE_1.0",
    "cat__Osteoporos_inkl_Osteopenie_T_Wert_abhängig_0.0",
    "cat__currsmo0_1.0",
    "num__Cer_d18_16",
    "num__Cer_d18_24_0",
    "num__GESAMT_A",
    "num__CERT2",
]

NA_VALUES = ["", " ", "NA", "N/A", "NULL"]

In [2]:
def _select_top_features(df: pd.DataFrame) -> pd.DataFrame:
    missing = [col for col in TOP_TRANSFORMED_FEATURES if col not in df.columns]
    if missing:
        raise KeyError(
            "The preprocessor did not create the expected encoded features: "
            f"{missing}"
        )
    return df[TOP_TRANSFORMED_FEATURES]


def build_pipeline() -> Pipeline:
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )

    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", categorical_pipe, SELECTED_CATEGORICAL),
            ("num", numeric_pipe, SELECTED_NUMERIC),
        ],
        remainder="drop",
    )
    preprocessor.set_output(transform="pandas")

    feature_selector = FunctionTransformer(_select_top_features, validate=False)

    xgb_params = dict(
        tree_method="hist",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("feature_selector", feature_selector),
            ("classifier", XGBClassifier(**xgb_params)),
        ]
    )

In [3]:
data = pd.read_csv(DATA_PATH, sep=";", na_values=NA_VALUES)
data = data[data[TARGET_COLUMN] != 1].reset_index(drop=True)

y = data[TARGET_COLUMN].replace({2: 1})
feature_frame = data[SELECTED_CATEGORICAL + SELECTED_NUMERIC].copy()
for column in SELECTED_NUMERIC:
    numeric_series = pd.to_numeric(feature_frame[column], errors="coerce")
    median = numeric_series.median()
    if pd.isna(median):
        median = 0.0
    feature_frame[column] = numeric_series.fillna(median)

X_train, X_test, y_train, y_test = train_test_split(
    feature_frame,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

pipeline = build_pipeline()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=1,
)
print(
    f"5-fold CV balanced accuracy: mean={cv_scores.mean():.4f} "
    f"(std={cv_scores.std():.4f})"
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(f"Test balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("Classification report:", classification_report(y_test, y_pred))
print("Features used for training:")
print(TOP_TRANSFORMED_FEATURES)

/var/folders/rp/sz_1shj9783116jtrj1kc04m0000gn/T/ipykernel_47945/3506649016.py:1: DtypeWarning: Columns (3,32,119,121,125,129,231,330,342,588) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(DATA_PATH, sep=";", na_values=NA_VALUES)


5-fold CV balanced accuracy: mean=0.5147 (std=0.0167)
Test balanced accuracy: 0.5084
Test ROC AUC: 0.4891
Classification report:               precision    recall  f1-score   support

           0       0.29      0.05      0.09        73
           1       0.79      0.96      0.86       263

    accuracy                           0.76       336
   macro avg       0.54      0.51      0.48       336
weighted avg       0.68      0.76      0.70       336

Features used for training:
['cat__ANY_OPO_1.0', 'cat__Osteoporos_inkl_Osteopenie_T_Wert_abhängig_1.0', 'cat__ANY_OPO_0.0', 'cat__ANY_OPE_1.0', 'cat__Osteoporos_inkl_Osteopenie_T_Wert_abhängig_0.0', 'cat__currsmo0_1.0', 'num__Cer_d18_16', 'num__Cer_d18_24_0', 'num__GESAMT_A', 'num__CERT2']
